#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType

#1. Read from bronze layer

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")
display(df)

#2. silver transformation

##2.1 Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##2.2 Normalization and cleaning of Country

In [0]:
country = {
        "Australia"
        , "United states"
        , "Canada"        
        , "United Kingdom"
        , "France"
        , "Germany"
        , "n/a"
        }
df = (
        df
        .withColumn("CNTRY"
                   , F.when(col("CNTRY").isin("US", "USA")  , "United states") 
                   .when(col("CNTRY") == "DE", "Germany")                                      
                   .otherwise(col("CNTRY"))
                   
        )
        .withColumn("CNTRY"
                    , F.when(col("CNTRY").isin(country), col("CNTRY"))
                    .otherwise("n/a")
                    )
)

df.select("CNTRY").distinct().display()

##2.3 Cleaning Customer ID

In [0]:
df = df.withColumn("CID", F.regexp_replace(col("CID"), "-", ""))

##2.4 Rename column

In [0]:
Renamed_Map = {
    "CID": "customer_number"
    , "CNTRY": "country"
}

for old_name, new_name in Renamed_Map.items():
    df = df.withColumnRenamed(old_name, new_name)

##2.4 dataframe sanity check

In [0]:
df.limit(10).display()

#3. wirte dataframe to silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_customer_location")

##3.1 sanity chack for silver table

In [0]:
%sql
select * from workspace.silver.erp_customer_location limit 10